In [1]:
import pandas as pd
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.config import TRAIN_PARQUET_PATH
from src.generation.alignment_benchmarks import stable_token_hash
import random
import re
from tqdm import tqdm
from gensim.models import Word2Vec
from transformers import pipeline
import warnings
warnings.filterwarnings("ignore")
import torch

c:\Users\thioy\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
# =========================
# 1. N-GRAM HELPERS
# =========================
def tokenize(text):
    return re.findall(r"\w+(?:[-’']\w+)*|[^\w\s]", str(text), flags=re.UNICODE)


def word_tokens(text):
    return re.findall(r"\w+(?:[-’']\w+)*", str(text).lower(), flags=re.UNICODE)


def make_ngrams(tokens, max_n=3):
    out = []

    for n in range(1, max_n + 1):
        for i in range(len(tokens) - n + 1):
            ng = tokens[i:i+n]
            out.append("_".join(ng))

    return out


def detokenize_ngram(ngram):
    return ngram.replace("_", " ")


# =========================
# 2. TRAIN CMDR N-GRAM EMBEDDINGS
# =========================
def train_cmdr_ngram_embeddings(df, wolof_col="wolof", french_col="french",
                                 max_n=3, vector_size=150, window=10,
                                 min_count=2, epochs=20, seed=42):
    random.seed(seed)
    corpus = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Building n-gram corpus"):
        wo = word_tokens(row[wolof_col])
        fr = word_tokens(row[french_col])
        wo_ngrams = make_ngrams(wo, max_n=max_n)
        fr_ngrams = make_ngrams(fr, max_n=max_n)

        # Keep order: Wolof first, French second, with overlap in window
        # This way only genuinely aligned n-grams share context
        sentence = wo_ngrams + fr_ngrams  # NO shuffle
        corpus.append(sentence)

    model = Word2Vec(sentences=corpus, vector_size=vector_size, window=window,
                     min_count=min_count, workers=1, epochs=epochs, seed=seed,
                     hashfxn=stable_token_hash)
    return model


# =========================
# 3. VOCAB FILTERS
# =========================
from collections import Counter
import math


def build_ngram_vocab_and_counts(series, max_n=3, min_len=3):
    vocab = set()
    counts = Counter()

    for text in series.astype(str):
        toks = word_tokens(text)

        for ng in make_ngrams(toks, max_n=max_n):
            if len(ng.replace("_", "")) >= min_len:
                vocab.add(ng)
                counts[ng] += 1

    return vocab, counts

# =========================
# 4. FIND BEST FRENCH N-GRAM
# =========================
def ngram_size(ngram):
    return len(ngram.split("_"))


def best_french_neighbor(
    wolof_ngram,
    model,
    french_vocab,
    french_counts,
    threshold=0.65,
    topn=50,
    min_freq=20,
    same_ngram_size=True
):
    if wolof_ngram not in model.wv:
        return None, 0.0

    src_n = ngram_size(wolof_ngram)

    candidates = []

    for cand, score in model.wv.most_similar(wolof_ngram, topn=topn):
        if cand not in french_vocab:
            continue

        if cand == wolof_ngram:
            continue

        if score < threshold:
            continue

        if french_counts[cand] < min_freq:
            continue

        if same_ngram_size and ngram_size(cand) != src_n:
            continue

        rank_score = score * math.log1p(french_counts[cand])

        candidates.append((cand, score, rank_score))

    if not candidates:
        return None, 0.0

    candidates.sort(key=lambda x: x[2], reverse=True)

    best_cand, best_score, _ = candidates[0]
    return best_cand, best_score


# =========================
# 5. PROTECTION
# =========================
def mask_protected(text, entities=None):
    spans = []

    if entities:
        for ent in entities:
            spans.append((ent["start"], ent["end"]))

    for m in re.finditer(r"\b[A-ZÀ-Ý0-9]{2,}\b", text):
        spans.append((m.start(), m.end()))

    for m in re.finditer(r"\b\d+(?:[.,]\d+)?\b", text):
        spans.append((m.start(), m.end()))

    spans = sorted(spans)
    merged = []

    for s, e in spans:
        if not merged or s > merged[-1][1]:
            merged.append([s, e])
        else:
            merged[-1][1] = max(merged[-1][1], e)

    saved = {}
    masked = text

    for i, (s, e) in enumerate(reversed(merged)):
        token = f"__protected_{i}__"
        saved[token] = masked[s:e]
        masked = masked[:s] + token + masked[e:]

    return masked, saved


def restore_protected(text, saved):
    for token, value in saved.items():
        text = text.replace(token, value)
    return text


# =========================
# 6. WOLOF NORMALIZATION
# =========================
rules = [
    (r"(\w+)-a-(\w+)", r"\1a\2"),
    (r"-", r" "),
    (r"([aieo])\1", r"\1"),
    (r"uu", r"__OU__"),
    (r"ñ+", r"gn"),
    (r"ŋ+", r"ng"),
    (r"ë+", r"__EU__"),
    (r"ée", r"e"),
    (r"g([ieéèêë])", r"g__U__\1"),
    (r"ó", r"o"),
    (r"(à|á|â|ä)", r"a"),
    (r"u([blt]+)", r"__OU__\1"),
    (r"u+", r"__OU__"),
    (r"q", r"kh"),
    (r"x", r"kh"),
    (r"c([aeiouy]+)", r"th\1"),
    (r"c{2}\b", r"thie"),
    (r"[Jj]([eaoë]{1,2})", r"di\1"),
    (r"[Jj]([i]+)", r"dj\1"),
    (r"[Jj]([u]+)", r"dio\1"),
    (r"th([aeouy]+)", r"thi\1"),
    (r"e\b", r"é"),
    (r"n\b", r"ne"),
    (r"gg", r"g"),
    (r"__EU__", r"eu"),
    (r"__OU__", r"ou"),
    (r"__U__", r"u")
]


def normalize_wolof_token(token):
    out = token.lower()
    for pattern, repl in rules:
        out = re.sub(pattern, repl, out)
    return out


from collections import defaultdict

def build_alignment_table(df, wolof_col="wolof", french_col="french", min_count=3):
    """
    For each Wolof unigram, collect which French unigrams appear
    in its aligned sentence. High co-occurrence = plausible translation.
    """
    cooc = defaultdict(Counter)
    for _, row in df.iterrows():
        wo_toks = set(word_tokens(row[wolof_col]))
        fr_toks = set(word_tokens(row[french_col]))
        for w in wo_toks:
            for f in fr_toks:
                cooc[w][f] += 1

    # Normalize to keep only frequent alignments
    alignment = {}
    for w, fr_counts in cooc.items():
        total = sum(fr_counts.values())
        alignment[w] = {f: c/total for f, c in fr_counts.items()
                        if c >= min_count}
    return alignment

def ngram_alignment_score(wolof_ngram, french_ngram, alignment_table):
    """
    Average alignment probability across constituent unigrams.
    Returns 0 if no alignment evidence exists.
    """
    wo_toks = wolof_ngram.split("_")
    fr_toks = french_ngram.split("_")
    scores = []
    for wt in wo_toks:
        if wt in alignment_table:
            # max alignment score to any French token in the ngram
            best = max((alignment_table[wt].get(ft, 0) for ft in fr_toks), default=0)
            scores.append(best)
        else:
            scores.append(0)
    return sum(scores) / len(scores) if scores else 0


def positional_penalty(wolof_pos_ratio, french_candidate_pos_ratio, sigma=0.35):
    """
    Gaussian penalty: candidates far from the Wolof token's
    relative position in the sentence are penalized.
    """
    import math
    diff = abs(wolof_pos_ratio - french_candidate_pos_ratio)
    return math.exp(-(diff ** 2) / (2 * sigma ** 2))


def best_french_neighbor_from_paired_sentence(
    wolof_ngram,
    paired_french_text,
    model,
    wolof_pos_ratio=None,       # <-- position of this ngram in Wolof sentence
    alignment_table=None,
    threshold=0.70,
    alignment_threshold=0.01,
    max_n=3
):
    if wolof_ngram not in model.wv:
        return None, 0.0

    fr_tokens = word_tokens(paired_french_text)
    fr_len = len(fr_tokens)
    paired_candidates = make_ngrams(fr_tokens, max_n=max_n)
    scored = []

    for idx, cand in enumerate(paired_candidates):
        if cand not in model.wv:
            continue
        if len(cand.split("_")) != len(wolof_ngram.split("_")):
            continue

        embed_score = model.wv.similarity(wolof_ngram, cand)
        if embed_score < threshold:
            continue

        # Alignment guard
        if alignment_table is not None:
            align_score = ngram_alignment_score(wolof_ngram, cand, alignment_table)
            if align_score < alignment_threshold:
                continue

        # Positional penalty
        pos_score = 1.0
        if wolof_pos_ratio is not None and fr_len > 0:
            fr_pos_ratio = idx / fr_len
            pos_score = positional_penalty(wolof_pos_ratio, fr_pos_ratio)

        combined = embed_score * (1 + (align_score if alignment_table else 0)) * pos_score
        scored.append((cand, embed_score, combined))

    if not scored:
        return None, 0.0

    scored.sort(key=lambda x: x[2], reverse=True)
    return scored[0][0], scored[0][1]
# =========================
# 7. CMDR-STYLE TOP-K SUBSTITUTION
# =========================
def cmdr_substitute_wolof_to_french(
    wolof_text, paired_french_text, model,
    alignment_table=None,
    max_n=2,
    num_substitutions=2,
    threshold=0.70
):
    tokens = word_tokens(wolof_text)
    wo_len = len(tokens)
    candidates = []

    n_penalty = {1: 1.0, 2: 0.85, 3: 0.70}

    for n in range(max_n, 0, -1):
        for i in range(len(tokens) - n + 1):
            src_ngram = "_".join(tokens[i:i+n])

            # Position of this ngram's midpoint in the Wolof sentence
            pos_ratio = (i + n / 2) / wo_len if wo_len > 0 else 0.5

            tgt_ngram, score = best_french_neighbor_from_paired_sentence(
                wolof_ngram=src_ngram,
                paired_french_text=paired_french_text,
                model=model,
                wolof_pos_ratio=pos_ratio,      
                alignment_table=alignment_table,
                threshold=threshold,
                max_n=max_n
            )

            if tgt_ngram:
                adjusted_score = score * n_penalty.get(n, 0.7)
                candidates.append({
                    "start": i, "end": i+n,
                    "src": src_ngram, "tgt": tgt_ngram,
                    "score": adjusted_score, "n": n
                })

    candidates = sorted(
        candidates,
        key=lambda x: (x["score"], x["n"]),
        reverse=True
    )

    selected = []
    occupied = set()

    for cand in candidates:
        span = set(range(cand["start"], cand["end"]))

        if span & occupied:
            continue

        selected.append(cand)
        occupied.update(span)

        if len(selected) >= num_substitutions:
            break

    selected = sorted(selected, key=lambda x: x["start"])

    output = []
    inserted_french = set()
    i = 0

    while i < len(tokens):
        match = next((c for c in selected if c["start"] == i), None)

        if match:
            fr_text = detokenize_ngram(match["tgt"])
            output.append(fr_text)
            inserted_french.update(word_tokens(fr_text))
            i = match["end"]
        else:
            output.append(tokens[i])
            i += 1

    return " ".join(output), inserted_french, selected


# =========================
# 8. FULL PIPELINE
# =========================
def process_cmdr_paper_style(
    wolof_text,
    paired_french_text,
    entities,
    model,
    max_n=3,
    num_substitutions=2,
    threshold=0.55,
    alignment_table=None
):
    if not isinstance(wolof_text, str) or not wolof_text.strip():
        return wolof_text

    # 1. Protect entities before substitutions
    masked, saved = mask_protected(wolof_text, entities)

    # 2. Substitute Wolof n-grams using ONLY candidates from paired French sentence
    mixed, inserted_french, selected = cmdr_substitute_wolof_to_french(
        wolof_text=masked,
        paired_french_text=paired_french_text,
        model=model,
        max_n=max_n,
        num_substitutions=num_substitutions,
        threshold=threshold,
        alignment_table=alignment_table
    )

    # 3. Normalize only non-French, non-protected Wolof tokens
    final = []

    for tok in mixed.split():
        clean = re.sub(r"[^\wÀ-ÿ’'-]", "", tok).lower()

        if re.fullmatch(r"__protected_\d+__", tok, flags=re.IGNORECASE):
            final.append(tok.lower())
        elif clean in inserted_french:
            final.append(tok)
        else:
            final.append(normalize_wolof_token(tok))

    result = " ".join(final)

    # 4. Restore protected spans
    result = restore_protected(result, saved)

    result = re.sub(r"\s+", " ", result).strip()
    return result

In [3]:
# =========================
# 9. RUN
# =========================
print("Loading dataset...")
parallel_df = pd.read_parquet(TRAIN_PARQUET_PATH)

print("Training CMDR n-gram embeddings...")
cmdr_model = train_cmdr_ngram_embeddings(
    parallel_df,
    wolof_col="wolof",
    french_col="french",
    max_n=3,
    min_count=2,
    epochs=20
)

print("Building French n-gram vocabulary...")
french_vocab, french_counts = build_ngram_vocab_and_counts(
    parallel_df["french"],
    max_n=3,
    min_len=3
)

# Optional NER protection. Keep this disabled unless all benchmark methods
# are regenerated with the same setting.
USE_NER = False
if USE_NER:
    device = 0 if torch.cuda.is_available() else -1
    print("Loading NER model...")
    ner_pipeline = pipeline(
        "token-classification",
        model="davlan/xlm-roberta-large-ner-hrl",
        aggregation_strategy="simple",
        device=device
    )
    def data_generator(df, column):
        for text in df[column]:
            yield str(text) if pd.notna(text) else ""
    print("Extracting entities...")
    all_entities = list(tqdm(
        ner_pipeline(data_generator(parallel_df, "wolof"), batch_size=32),
        total=len(parallel_df)
    ))
else:
    all_entities = [[] for _ in range(len(parallel_df))]

print("Building alignment table...")
alignment_table = build_alignment_table(parallel_df, min_count=3)

print("Generating CMDR-style code-mixed text...")
outputs = []

for wolof_text, french_text, entities in tqdm(
    zip(parallel_df["wolof"], parallel_df["french"], all_entities),
    total=len(parallel_df)
):
    out = process_cmdr_paper_style(
        wolof_text=wolof_text,
        paired_french_text=french_text,
        entities=entities,
        model=cmdr_model,
        alignment_table=alignment_table,  # pass it through
        max_n=2,
        num_substitutions=2,
        threshold=0.70
    )
    outputs.append(out)

parallel_df["informal code switched"] = outputs

print(parallel_df[["french", "wolof", "informal code switched"]].head(10))

Loading dataset...
Training CMDR n-gram embeddings...


Building n-gram corpus: 100%|██████████| 17777/17777 [00:01<00:00, 10664.60it/s]


Building French n-gram vocabulary...
Building alignment table...
Generating CMDR-style code-mixed text...


100%|██████████| 17777/17777 [01:29<00:00, 198.75it/s]

                                              french  \
0  Cette lettre doit être légalisée par le Minist...   
1           « On n’a pas eu le temps de se sauver.\n   
2  L’Écosse est tout à fait votre maison et nous ...   
3  Jotnanews: L’Espagne injecte 84 milliards FCFA...   
4                                     quel âge as-tu   
5  Abou Ousmane Sy, Mamadou Yoro Diallo et Ousman...   
6  Et comme l’on a coutume de le dire dans nos co...   
7  La Zambie est membre des Nations Unies, de l'U...   
8                                             humide   
9  Quelques centaines de personnes sont décédées ...   

                                               wolof  \
0  Bataaxal bii jëwriñu lu ajju ci mbiru bitim ré...   
1       "Amuñu woon benn jot ngir rawal sunu bopp.\n   
2      Ekost sa dëkk la te bëggna nu nga toog fi."\n   
3  Xibaari Jotna : Espaañ joxe na juróom-ñett-fuk...   
4                                    ñaata at nga am   
5  Abu Usmaan Si, Mamadu Yoro Jàllo ak Usmaan N

In [4]:
parallel_df.iloc[7].to_dict()

{'french': "La Zambie est membre des Nations Unies, de l'Union africaine et de la communauté du développement de l'Afrique australe (SADC).",
 'wolof': 'Zambie réew la mu bokk ci ONU, Union Africaine, ak Southern African Development Community (SADC).',
 'sources': 'Google SMOL',
 'informal code switched': 'zambie rew la mou bokk thi ONU ounione afrithiainé ak soouthierne afrithiane development thiommounity SADC'}

In [5]:
parallel_df.iloc[6].to_dict()


{'french': 'Et comme l’on a coutume de le dire dans nos contrées : Si tu ne sais pas où tu vas, retourne d’ où tu viens et c’est bien normal que nous revenons à la cage de départ pour le grands dame des consommateurs.\n',
 'wolof': 'Ni ñu ko tàmm a waxe fii : boo xamatul foo jëm, dellul fa nga jóge woon, baax na lool ñu dellu fa ñu jóge woon ngir bégal way-jëfandiku yi.\n',
 'sources': 'mafand',
 'informal code switched': 'ni gnou le dire a wakhé fi bo khamatoul fo jeum delloul fa nga cage de bakh na lol gnou dellou fa gnou diogué wone nguir bégal way jeufandikou yi'}

In [6]:
parallel_df.iloc[450].to_dict()

{'french': 'Nous avons tous les deux rempli un questionnaire et répondu à quelques questions sur nos antécédents de santé.',
 'wolof': 'Nu bokk joxe leeral yi nu laaj ci kesioneer bi, ba noppi tontu ay laaj ci jafe-jafe wérgi-yaram yu nu musa am.',
 'sources': 'Google SMOL',
 'informal code switched': 'nou bokk diokhé leral yi nou laj thi kesioner bi ba noppi tontou ay laj thi diafé diafé wérgui yaram you nou mousa am'}

In [7]:
parallel_df.iloc[1550].to_dict()

{'french': "Hé les gars, désolé de ne pas avoir rédigé le rapport, je ne suis pas rentré à la maison jusqu'à présent après l'entraînement.",
 'wolof': 'Sama gaa yi, mangi jeggalu ci niakk bind xaatim bi, dellu waatu ma ci kerr ga gannaaw bima jeffe ba leegui.',
 'sources': 'Google SMOL',
 'informal code switched': 'sama ga yi mangui diegalou thi niakk bind khatim bi dellou watou ma thi kerr ga gannaw bima dieffé ba legoui'}

## M3 benchmark finalization

This section freezes the CMDR resources and exports a deterministic benchmark dataset in the same schema as M0-M2. Every accepted substitution records its embedding, alignment, position, and combined scores.

In [8]:
from src.config import (
    CMDR_ALIGNMENT_PATH, CMDR_MODEL_PATH, M3_CMDR_MANIFEST_PATH,
    M3_CMDR_PATH, M3_CMDR_REVIEW_PATH, TRAIN_PARQUET_PATH,
)
from src.generation.alignment_benchmarks import (
    METHOD_M3, generate_m3_dataset, save_cmdr_resources,
    write_alignment_benchmark,
)

BENCHMARK_SEED = 2026
CMDR_TRAINING_SEED = 42  # default used by train_cmdr_ngram_embeddings above
M3_MAX_NGRAM = 2
M3_MAX_SUBSTITUTIONS = 2
M3_EMBEDDING_THRESHOLD = 0.70
M3_ALIGNMENT_THRESHOLD = 0.01
M3_POSITION_SIGMA = 0.35
REVIEW_SAMPLE_SIZE = 100

# Saving is quick; training cmdr_model above is the long-running step.
save_cmdr_resources(
    cmdr_model, alignment_table,
    model_path=CMDR_MODEL_PATH, table_path=CMDR_ALIGNMENT_PATH,
)
entities_by_row = {
    row_id: [(entity["start"], entity["end"]) for entity in entities]
    for row_id, entities in enumerate(all_entities)
}
m3_pairs = generate_m3_dataset(
    parallel_df, model=cmdr_model, alignment_table=alignment_table,
    seed=BENCHMARK_SEED, max_n=M3_MAX_NGRAM,
    max_substitutions=M3_MAX_SUBSTITUTIONS,
    embedding_threshold=M3_EMBEDDING_THRESHOLD,
    alignment_threshold=M3_ALIGNMENT_THRESHOLD,
    position_sigma=M3_POSITION_SIGMA,
    entities_by_row=entities_by_row,
)
assert len(m3_pairs) == len(parallel_df)
assert m3_pairs["source_id"].is_unique
assert m3_pairs["formal_wolof"].tolist() == parallel_df["wolof"].fillna("").astype(str).tolist()
m3_pairs.head()

,source_id,source_row,source_corpus,method,seed,informal_wolof,formal_wolof,changed,edit_count,alignment_edit_count,spelling_edit_count,error_types,applied_rules,alignment_edits
0,parallel:0,0,french_wolof_parallel,m3_cmdr_plus_beqi_rules,2026,batakhal bi jeuwrignou lou ajjou thi mbirou bi...,Bataaxal bii jëwriñu lu ajju ci mbiru bitim ré...,True,24,0,24,"[""beqi_inspired_orthographic_rules""]","{""a_diacritic_to_a"": 1, ""c_vowel_to_th"": 2, ""e...",[]
1,parallel:1,1,french_wolof_parallel,m3_cmdr_plus_beqi_rules,2026,"""amougnou wone benne diot nguir rawal sounou b...","""Amuñu woon benn jot ngir rawal sunu bopp.\n",True,11,0,11,"[""beqi_inspired_orthographic_rules""]","{""enye_to_gn"": 1, ""final_n_to_ne"": 2, ""j_eao_t...",[]
2,parallel:2,2,french_wolof_parallel,m3_cmdr_plus_beqi_rules,2026,"ekost sa deukk la té beugna nou nga tog fi.""","Ekost sa dëkk la te bëggna nu nga toog fi.""\n",True,7,0,7,"[""beqi_inspired_orthographic_rules""]","{""double_g_to_g"": 1, ""final_e_to_e_acute"": 1, ...",[]
3,parallel:3,3,french_wolof_parallel,m3_cmdr_plus_beqi_rules,2026,khibari diotna : espagne diokhé na jouroom gne...,Xibaari Jotna : Espaañ joxe na juróom-ñett-fuk...,True,28,0,28,"[""beqi_inspired_orthographic_rules""]","{""c_vowel_to_th"": 1, ""enye_to_gn"": 3, ""final_e...",[]
4,parallel:4,4,french_wolof_parallel,m3_cmdr_plus_beqi_rules,2026,gnata at nga am,ñaata at nga am,True,2,0,2,"[""beqi_inspired_orthographic_rules""]","{""enye_to_gn"": 1, ""reduce_double_vowel"": 1}",[]


In [9]:
m3_diagnostics = {
    "rows": len(m3_pairs),
    "changed_rows": int(m3_pairs["changed"].sum()),
    "changed_rate": float(m3_pairs["changed"].mean()),
    "rows_with_alignment": int(m3_pairs["alignment_edit_count"].gt(0).sum()),
    "alignment_edits": int(m3_pairs["alignment_edit_count"].sum()),
    "spelling_edits": int(m3_pairs["spelling_edit_count"].sum()),
}
m3_diagnostics

{'rows': 17777,
 'changed_rows': 17102,
 'changed_rate': 0.9620295887945097,
 'rows_with_alignment': 6176,
 'alignment_edits': 9548,
 'spelling_edits': 317451}

In [10]:
m3_manifest = write_alignment_benchmark(
    m3_pairs,
    output_path=M3_CMDR_PATH,
    review_path=M3_CMDR_REVIEW_PATH,
    manifest_path=M3_CMDR_MANIFEST_PATH,
    method=METHOD_M3,
    input_path=TRAIN_PARQUET_PATH,
    artifacts={"cmdr_word2vec": CMDR_MODEL_PATH, "cmdr_alignment_table": CMDR_ALIGNMENT_PATH},
    parameters={
        "benchmark_seed": BENCHMARK_SEED,
        "training_seed": CMDR_TRAINING_SEED,
        "stable_hash": "CRC32 UTF-8",
        "workers": 1,
        "max_ngram": M3_MAX_NGRAM,
        "max_substitutions": M3_MAX_SUBSTITUTIONS,
        "embedding_threshold": M3_EMBEDDING_THRESHOLD,
        "alignment_threshold": M3_ALIGNMENT_THRESHOLD,
        "position_sigma": M3_POSITION_SIGMA,
        "use_ner": USE_NER,
        "selection": "deterministic score with lexical tie-break",
        "postprocessing": "shared M1 Beqi-inspired spelling rules",
    },
    review_size=REVIEW_SAMPLE_SIZE,
    review_seed=BENCHMARK_SEED,
)
m3_manifest

{'method': 'm3_cmdr_plus_beqi_rules',
 'created_at_utc': '2026-08-03T16:30:29.725315+00:00',
 'input_path': 'C:\\Users\\thioy\\Desktop\\School\\PFE\\data\\translation data\\train-00000-of-00001.parquet',
 'input_sha256': 'bd4910104fc121ff6924ca9a8b1a0d6bdaa6f28601108703cc9b29b2dc9eb27e',
 'output_path': 'C:\\Users\\thioy\\Desktop\\School\\PFE\\data\\synthetic\\m3_cmdr_rules_pairs.csv',
 'output_sha256': '169b85e1184b4d0ca5f418c00beec2f1f7e178e68af92bf8fb77d47fd82f3f54',
 'rows': 17777,
 'changed_rows': 17102,
 'unchanged_rows': 675,
 'rows_with_alignment_edits': 6176,
 'alignment_edits': 9548,
 'spelling_edits': 317451,
 'artifacts': {'cmdr_word2vec': {'path': 'C:\\Users\\thioy\\Desktop\\School\\PFE\\artifacts\\cmdr_word2vec.model',
   'sha256': '4eff54f30c5c57858c957732a07f1d262cd4d3539deccf761c7d722ae8227067',
   'sidecars': [{'path': 'C:\\Users\\thioy\\Desktop\\School\\PFE\\artifacts\\cmdr_word2vec.model.syn1neg.npy',
     'sha256': '26d35922c4c131503697dc0558c35cd9d7e1f5daec1961766